In [ ]:
# Step 1 — Import libraries
%pip install matplotlib seaborn scikit-learn pandas numpy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

print("All libraries imported successfully!")


^C
Note: you may need to restart the kernel to use updated packages.


Matplotlib is building the font cache; this may take a moment.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.3 MB 3.7 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/9.3 MB 4.4 MB/s eta 0:00:02
   ---------- ----------------------------- 2.4/9.3 MB 3.7 MB/s eta 0:00:02
   ------------- -------------------------- 3.1/9.3 MB 3.6 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.3 MB 3.5 MB/s eta 0:00:02
   ------------------- -------------------- 4.5/9.3 MB 3.6 MB/s eta 0:00:02
   ---------------------- ----------------- 5.2/9.3 MB 3.5 MB/s eta 0:00:02
   ----

In [ ]:
# Step 2 — Load dataset
df = pd.read_csv('StudentsPerformance.csv')
print("Shape:", df.shape)
df.head()


In [ ]:
# Step 3 — Exploratory Data Analysis (EDA)
print("Dataset Info:")
print(df.info())
print("\nNull values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
df.describe()


In [ ]:
# Step 4 — Visualizations

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Score distributions
for i, col in enumerate(['math score', 'reading score', 'writing score']):
    axes[i].hist(df[col], bins=20, color=['#4C72B0', '#DD8452', '#55A868'][i], edgecolor='white')
    axes[i].set_title(f'Distribution: {col}')
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Count')

plt.suptitle('Score Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved!")


In [ ]:
# Step 5 — Correlation Heatmap
plt.figure(figsize=(6, 4))
corr = df[['math score', 'reading score', 'writing score']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Correlation Between Scores')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Heatmap saved!")


In [ ]:
# Step 6 — Gender vs Average Math Score
gender_avg = df.groupby('gender')['math score'].mean()
gender_avg.plot(kind='bar', color=['#4C72B0', '#DD8452'], edgecolor='white', figsize=(5, 4))
plt.title('Average Math Score by Gender')
plt.ylabel('Average Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('gender_vs_score.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Step 7 — Data Preprocessing

df_model = df.copy()

# Encode all categorical columns
le = LabelEncoder()
cat_cols = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']

for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

# Create target: 1 = Pass (math score >= 50), 0 = Fail
df_model['pass_fail'] = (df_model['math score'] >= 50).astype(int)

print("Class distribution:")
print(df_model['pass_fail'].value_counts())
print(f"\nPass rate: {df_model['pass_fail'].mean()*100:.1f}%")


In [ ]:
# Step 8 — Feature Selection & Train-Test Split

features = ['gender', 'race/ethnicity', 'parental level of education', 
            'lunch', 'test preparation course', 'reading score', 'writing score']

X = df_model[features]
y = df_model['pass_fail']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print(f"Features used:    {len(features)}")


In [ ]:
# Step 9 — Train All Classifiers

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes':         GaussianNB(),
    'SVM':                 SVC(kernel='rbf', random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name:<22} Accuracy: {acc*100:.2f}%")


In [ ]:
# Step 10 — Detailed Report for Best Model

best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test)

print(f"Best Model: {best_model_name} ({results[best_model_name]*100:.2f}% accuracy)\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Fail', 'Pass']))


In [ ]:
# Step 11 — Model Comparison Chart

names = list(results.keys())
accuracies = [v * 100 for v in results.values()]
colors = ['#4C72B0' if n != best_model_name else '#2ecc71' for n in names]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, accuracies, color=colors, edgecolor='white', width=0.6)
plt.ylim(80, 100)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.xticks(rotation=15, ha='right')

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nBest model: {best_model_name} with {max(accuracies):.2f}% accuracy")


In [ ]:
# Step 12 — Feature Importance (Random Forest)

rf_model = models['Random Forest']
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({'Feature': features, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(7, 4))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='#4C72B0', edgecolor='white')
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
